# 04 — Final Predictions
**DataStorm v7.0 | SkyNet Team**

Goal: Generate the final submission CSV with `Outlet_ID` and `Maximum_Monthly_Liters` for January 2026.

The two-stage uncapping model has produced:
- `jan_2026_potential`: Base XGBoost prediction × constraint_multiplier × seasonality_index
- With protective floor (≥ historical max) and peer-group ceiling (sanity bound)

In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

GOLD_DIR = ROOT / 'data' / 'gold'
OUTPUTS  = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

print('Setup complete')

Setup complete


In [4]:
# Load potential demand results from the modeling notebook
results = pd.read_csv(GOLD_DIR / 'potential_demand_results.csv')
print('Results loaded:', results.shape)

# Select and rename columns for submission
# Output column: Maximum_Monthly_Liters (January 2026 latent potential)
submission = results[['Outlet_ID', 'jan_2026_potential']].copy()
submission.columns = ['Outlet_ID', 'Maximum_Monthly_Liters']

# Round to 2 decimal places
submission['Maximum_Monthly_Liters'] = submission['Maximum_Monthly_Liters'].round(2)

# ── Ensure no missing outlets ────────────────────────────────────────────────
master = pd.read_csv(ROOT / 'data' / 'silver' / 'outlet_master_clean.csv')
all_ids = master['Outlet_ID'].unique()
existing_ids = set(submission['Outlet_ID'])
missing_ids = set(all_ids) - existing_ids

if missing_ids:
    print(f'Warning: {len(missing_ids)} outlets missing from predictions.')
    # For missing outlets, use the median of their outlet type peer group
    # (more robust than global mean)
    if 'Outlet_Type' in master.columns:
        # Get outlet types for missing IDs
        missing_master = master[master['Outlet_ID'].isin(missing_ids)][['Outlet_ID', 'Outlet_Type']].copy()
        # Compute median potential by outlet type from existing predictions
        type_medians = results.groupby(
            [c for c in results.columns if c.startswith('Outlet_Type_')]
        )['jan_2026_potential'].median()
        # Fallback: use overall median
        fallback_val = submission['Maximum_Monthly_Liters'].median()
        missing_df = pd.DataFrame({
            'Outlet_ID': list(missing_ids),
            'Maximum_Monthly_Liters': fallback_val
        })
    else:
        fallback_val = submission['Maximum_Monthly_Liters'].median()
        missing_df = pd.DataFrame({
            'Outlet_ID': list(missing_ids),
            'Maximum_Monthly_Liters': fallback_val
        })
    submission = pd.concat([submission, missing_df], ignore_index=True)
    print(f'Filled missing outlets with median value: {fallback_val:.2f}')
else:
    print('All outlets accounted for.')

# ── Final Validation ──────────────────────────────────────────────────────────
print(f'\nSubmission preview:')
print(submission.head(10))
print(f'\nJanuary 2026 Forecast Statistics:')
print(submission['Maximum_Monthly_Liters'].describe().round(2))

# ── Save ──────────────────────────────────────────────────────────────────────
team_name = 'SkyNet'
submission_file = OUTPUTS / f'{team_name}_predictions.csv'
submission.to_csv(submission_file, index=False)

print(f'\nFinal submission file created: {submission_file}')
print(f'Total records in submission: {len(submission)}')

Results loaded: (19804, 81)
All outlets accounted for.

Submission preview:
   Outlet_ID  Maximum_Monthly_Liters
0  OUT_00001                  716.73
1  OUT_00002                  291.62
2  OUT_00003                  294.08
3  OUT_00004                  740.74
4  OUT_00005                  692.17
5  OUT_00006                  692.17
6  OUT_00007                 1025.89
7  OUT_00008                 1081.35
8  OUT_00009                  711.51
9  OUT_00010                  296.14

January 2026 Forecast Statistics:
count    19804.00
mean       433.14
std        515.21
min         32.21
25%        129.44
50%        198.56
75%        393.64
max       2800.02
Name: Maximum_Monthly_Liters, dtype: float64

Final submission file created: C:\Users\User\Documents\Projects\Datastorm\SkyNet-datastorm-v7\outputs\SkyNet_predictions.csv
Total records in submission: 19804
